# 🏦 SME Financial Risk Scoring — Model Training
**Stack:** Scikit-learn · Random Forest · SHAP · Joblib  
**Output:** `models/model.pkl`

## 1 · Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import warnings
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
np.random.seed(42)
print('All imports OK ✓')

## 2 · Synthetic Dataset Generation

> **Note:** Replace this section with your real dataset CSV when available.
> Just load it with `pd.read_csv('path/to/your_data.csv')` and skip to Section 3.

In [ ]:
N = 2000  # increase to 5000+ for better generalisation

df = pd.DataFrame({
    # Business profile
    'business_age':    np.random.randint(1, 30, N).astype(float),
    'industry_type':   np.random.randint(0, 5, N).astype(float),   # 0=Retail … 4=Textile
    'num_employees':   np.random.randint(1, 200, N).astype(float),
    'city_tier':       np.random.randint(1, 4, N).astype(int),      # 1=Metro, 2, 3=Rural
    # Financials
    'annual_revenue':  np.random.randint(500_000, 50_000_000, N).astype(float),
    'gst_consistency': np.random.uniform(30, 100, N),
    'existing_debt':   np.random.randint(0, 5_000_000, N).astype(float),
    'upi_volume':      np.random.randint(10_000, 2_000_000, N).astype(float),
    'utility_payment': np.random.randint(0, 3, N).astype(int),      # 0=Poor, 1=Avg, 2=Good
    # Loan request
    'loan_amount':     np.random.randint(100_000, 5_000_000, N).astype(float),
    'loan_duration':   np.random.randint(6, 60, N).astype(float),
})

print(df.shape)
df.head(3)

### 2a · Domain-Driven Risk Label Assignment

In [ ]:
def assign_risk(row):
    """
    Scoring heuristic (replace with real labels in production).
    Positive factors ↑ score → Lower risk.
    Negative factors ↓ score → Higher risk.
    Thresholds:  score > 5.5 → Low (0)
                 score > 2.5 → Medium (1)
                 else        → High (2)
    """
    s = 0.0

    # ── Positive signals ──────────────────────────────────────────
    s += min(float(row['business_age']) / 5.0, 3.0)          # older biz → safer (max +3)

    up = int(row['utility_payment'])
    s += 2.0 if up == 2 else (1.0 if up == 1 else 0.0)      # Good pay → +2

    s += (float(row['gst_consistency']) - 30.0) / 35.0       # 30-100% → 0-2
    s += min(float(row['annual_revenue']) / 10_000_000.0, 2.0)  # revenue cap +2
    s += min(float(row['upi_volume'])    / 500_000.0,     1.0)  # digital txns +1

    ct = int(row['city_tier'])
    s += 0.5 if ct == 1 else (0.2 if ct == 2 else 0.0)      # metro premium
    s += min(float(row['num_employees']) / 50.0, 1.0)        # scale proxy +1

    # ── Negative signals ──────────────────────────────────────────
    debt_ratio = float(row['existing_debt']) / max(float(row['annual_revenue']), 1.0)
    s -= min(debt_ratio * 3.0, 3.0)                          # high debt burden → -3

    loan_ratio = float(row['loan_amount']) / max(float(row['annual_revenue']), 1.0)
    s -= min(loan_ratio * 2.0, 2.0)                          # over-leveraged → -2

    return 0 if s > 5.5 else (1 if s > 2.5 else 2)

df['risk_label'] = df.apply(assign_risk, axis=1)

print('Label distribution:')
print(df['risk_label'].value_counts().rename({0:'Low', 1:'Medium', 2:'High'}))

## 3 · Train / Test Split

In [ ]:
FEATURES = [
    'business_age', 'industry_type', 'num_employees', 'city_tier',
    'annual_revenue', 'gst_consistency', 'existing_debt', 'upi_volume',
    'utility_payment', 'loan_amount', 'loan_duration'
]

X = df[FEATURES]
y = df['risk_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 4 · Train Random Forest

In [ ]:
clf = RandomForestClassifier(
    n_estimators   = 200,
    max_depth      = 12,
    min_samples_split = 5,
    class_weight   = 'balanced',   # handles label imbalance
    random_state   = 42,
    n_jobs         = -1,
)

clf.fit(X_train, y_train)
print('Training complete ✓')

## 5 · Evaluation

In [ ]:
y_pred = clf.predict(X_test)

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print(classification_report(y_test, y_pred, target_names=['Low','Medium','High']))

In [ ]:
# Cross-validation (5-fold stratified)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')
print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['Low','Medium','High']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 6 · Feature Importance (Random Forest built-in)

In [ ]:
importances = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance (Mean Decrease Impurity)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 7 · SHAP Analysis

In [ ]:
# Build TreeExplainer (fast, exact for tree models)
explainer = shap.TreeExplainer(clf)

# SHAP on test set sample (200 rows for speed)
X_sample = X_test.sample(200, random_state=42)
shap_values = explainer.shap_values(X_sample)   # shape: (200, 11, 3)

print(f'SHAP values shape: {np.array(shap_values).shape}')  # (n_samples, n_features, n_classes)

In [ ]:
# Summary plot — High Risk class (class 2)
shap.summary_plot(
    shap_values[:, :, 2],   # class 2 = High risk
    X_sample,
    feature_names=FEATURES,
    plot_type='bar',
    show=True
)

In [ ]:
# Beeswarm plot for richer view
shap.summary_plot(
    shap_values[:, :, 2],
    X_sample,
    feature_names=FEATURES,
    show=True
)

## 8 · Single-Sample Prediction Demo

In [ ]:
sample = {
    'business_age':    8.0,
    'industry_type':   2,       # Services
    'num_employees':   25,
    'city_tier':       1,       # Metro
    'annual_revenue':  8_000_000,
    'gst_consistency': 82.0,
    'existing_debt':   500_000,
    'upi_volume':      400_000,
    'utility_payment': 2,       # Good
    'loan_amount':     2_000_000,
    'loan_duration':   36,
}

arr = np.array([[sample[f] for f in FEATURES]])

pred_class = clf.predict(arr)[0]
proba      = clf.predict_proba(arr)[0]
label_map  = {0: 'Low 🟢', 1: 'Medium 🟡', 2: 'High 🔴'}

print(f'Predicted Risk : {label_map[pred_class]}')
print(f'Confidence     : {proba[pred_class]*100:.1f}%')
print(f'Probabilities  : Low={proba[0]*100:.1f}%  Medium={proba[1]*100:.1f}%  High={proba[2]*100:.1f}%')

In [ ]:
# Force plot for this sample
shap.initjs()
sv_single = explainer.shap_values(arr)
shap.force_plot(
    explainer.expected_value[pred_class],
    sv_single[0, :, pred_class],
    arr[0],
    feature_names=FEATURES
)

## 9 · Save Model

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(clf, '../models/model.pkl')
print('✅  Model saved to ../models/model.pkl')

# Verify round-trip
loaded = joblib.load('../models/model.pkl')
assert (loaded.predict(arr) == clf.predict(arr)).all(), 'Round-trip mismatch!'
print('✅  Load + predict round-trip verified')

## ✅ Checklist
- [ ] Dataset loaded / generated  
- [ ] Labels assigned correctly  
- [ ] Accuracy ≥ 85%  
- [ ] SHAP values computed  
- [ ] `model.pkl` saved  
- [ ] FastAPI `/predict` tested with Postman / curl